# DistilBERT, re-run on the project's standard 20% split

**Why this notebook exists.** `03_distilbert.ipynb` is Ryan's original run and
scores **0.9487 accuracy / 0.8951 macro F1** - the best number in the project.
But it held out **15%** of `train.json` for validation, while
`src/run_baseline.py` and `02_svm_mlp.ipynb` both hold out **20%**. The models are
therefore scored on different reviews, and the report cannot honestly put them in
one table.

This notebook runs the *same* model and the *same* hyperparameters at
`test_size=0.2`, so DistilBERT lands on the identical 1,481-review split as
everything else. It writes `results/distilbert_metrics.json`, which
`src/compare_models.py` picks up automatically and promotes to a fully
comparable row.

**Run this on a GPU.** `Runtime` -> `Change runtime type` -> **T4 GPU**. About two
minutes per epoch on a T4, against 30-60 minutes per epoch on CPU.

Nothing here is tuned. Changing a hyperparameter would defeat the purpose: the
only difference from `03_distilbert.ipynb` is the split.

**What this run records** (nothing here changes the training): per-epoch train and validation loss, accuracy, macro F1 and negative F1; training, validation and test timings; per-review probabilities for the validation and test sets; and three figures - learning curves, confusion matrix, and confidence of right vs wrong predictions. Everything lands in `results/` and is downloaded as one zip at the end.

## 1. Environment

In [ ]:
# Colab already has torch. Locally, install the CUDA build rather than the
# CPU one in requirements.txt, or this takes most of an hour per epoch.
%pip install -q transformers scikit-learn pandas

In [ ]:
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    get_linear_schedule_with_warmup,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
if DEVICE.type != 'cuda':
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

## 2. Settings

Identical to `03_distilbert.ipynb` except for `TEST_SIZE`, which is the whole
point of this notebook.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 256          # 98.8% of reviews fit whole - see notebooks/01_explore_data.ipynb
BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
TEST_SIZE = 0.2        # <-- the only change from 03_distilbert.ipynb (was 0.15)

## 3. Data

On Colab the next cell opens a file picker: select **both** `train.json` and
`test.json`. The repo is private, so it cannot simply be cloned here. Run locally,
the notebook finds `data/` from the repo root or from `notebooks/`.

In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not (Path('train.json').exists() and Path('test.json').exists()):
    print('Select BOTH train.json and test.json in the file picker.')
    files.upload()

In [ ]:
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'IE4483-sentiment-project']
ROOT = next((p for p in candidates if (p / 'data' / 'train.json').exists()), None)

if ROOT is None:
    # Colab fallback: the two json files sitting beside the notebook.
    if Path('train.json').exists():
        DATA_DIR, RESULTS_DIR = Path('.'), Path('results')
    else:
        raise FileNotFoundError(
            'train.json not found. Re-run the upload cell and select both json files.'
        )
else:
    ROOT = ROOT.resolve()
    DATA_DIR, RESULTS_DIR = ROOT / 'data', ROOT / 'results'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
train_df = pd.read_json(DATA_DIR / 'train.json')
test_df = pd.read_json(DATA_DIR / 'test.json')
print('data:', DATA_DIR, '| results:', RESULTS_DIR)
print('train', train_df.shape, '| test', test_df.shape)

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['reviews'].fillna('').tolist(),
    train_df['sentiments'].tolist(),
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=train_df['sentiments'].tolist(),
)
print(f'{len(train_texts)} train / {len(val_texts)} validation')
print('This must read 5920 train / 1481 validation to match the other models.')
assert len(val_texts) == 1481, 'split does not match src/compare_models.py'

## 4. Tokenise

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels=None, max_len=MAX_LEN):
        self.texts = [str(t).strip() for t in texts]
        self.labels = labels
        self.encodings = tokenizer(
            self.texts, truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt',
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_loader = DataLoader(ReviewDataset(train_texts, train_labels),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ReviewDataset(val_texts, val_labels), batch_size=BATCH_SIZE)
test_loader = DataLoader(ReviewDataset(test_df['reviews'].tolist()),
                         batch_size=BATCH_SIZE)

## 5. Train

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(DEVICE)

# Question (f) wants this number: the parameter cost against 20,001 for the
# linear models on TF-IDF.
N_PARAMS = sum(p.numel() for p in model.parameters())
print(f'{N_PARAMS:,} parameters')

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

In [ ]:
from tqdm.auto import tqdm

BEST_PATH = 'best_distilbert_20split.pt'   # .pt is gitignored - never commit weights
best_val_f1 = 0.0
train_seconds = 0.0
history = []   # one row per epoch -> learning curves and question (c)

for epoch in range(EPOCHS):
    model.train()
    started = time.perf_counter()
    total_train_loss = 0.0

    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]', leave=False):
        optimizer.zero_grad()
        out = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
            labels=batch['labels'].to(DEVICE),
        )
        out.loss.backward()
        total_train_loss += out.loss.item()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

    epoch_train_seconds = time.perf_counter() - started
    train_seconds += epoch_train_seconds

    # Validation. Eval mode + no_grad: no dropout, no random numbers used, so this
    # logging cannot change the training run.
    model.eval()
    preds, truth, probs = [], [], []
    total_val_loss = 0.0
    val_started = time.perf_counter()
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]', leave=False):
            out = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
                labels=batch['labels'].to(DEVICE),
            )
            total_val_loss += out.loss.item()
            probs.extend(torch.softmax(out.logits, dim=1)[:, 1].cpu().numpy().tolist())
            preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy().tolist())
            truth.extend(batch['labels'].numpy().tolist())
    val_seconds = time.perf_counter() - val_started

    row = {
        'epoch': epoch + 1,
        'train_loss': total_train_loss / len(train_loader),
        'val_loss': total_val_loss / len(val_loader),
        'val_accuracy': accuracy_score(truth, preds),
        'val_macro_f1': f1_score(truth, preds, average='macro'),
        'val_negative_f1': f1_score(truth, preds, pos_label=0),
        'train_seconds': round(epoch_train_seconds, 1),
        'val_seconds': round(val_seconds, 1),
    }
    history.append(row)
    print(f"Epoch {row['epoch']}/{EPOCHS} | Train Loss: {row['train_loss']:.4f} "
          f"| Val Loss: {row['val_loss']:.4f} | Val Acc: {row['val_accuracy']:.4f} "
          f"| Val Macro F1: {row['val_macro_f1']:.4f} | Val Neg F1: {row['val_negative_f1']:.4f} "
          f"| {row['train_seconds']:.0f}s train")

    if row['val_macro_f1'] > best_val_f1:
        best_val_f1 = row['val_macro_f1']
        best_epoch = epoch + 1
        torch.save(model.state_dict(), BEST_PATH)
        best_preds, best_truth, best_probs = list(preds), list(truth), list(probs)

print(f'\nBest epoch: {best_epoch} | total training time {train_seconds:.0f}s '
      f'({train_seconds/60:.1f} min)')

## 6. Full metrics for the best epoch

Accuracy alone is not reportable here - predicting 'positive' for everything
scores 0.853. Per-class precision, recall and F1 are what questions (c) to (f)
are marked on.

In [ ]:
report = classification_report(
    best_truth, best_preds, target_names=['negative', 'positive'],
    output_dict=True, zero_division=0,
)
print(classification_report(best_truth, best_preds,
                            target_names=['negative', 'positive'], zero_division=0))
cm = confusion_matrix(best_truth, best_preds, labels=[0, 1])
print('Confusion matrix (rows = true, cols = predicted):')
print(cm)

## 6b. Save the per-epoch history and every validation prediction

`distilbert_history.csv` is the evidence for question (c) - how the model behaved
epoch by epoch. `distilbert_val_predictions.csv` holds every validation review with
its true label, prediction and probability, which is what question (e) needs.

In [ ]:
FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

history_df = pd.DataFrame(history)
history_df.to_csv(RESULTS_DIR / 'distilbert_history.csv', index=False, lineterminator='\n')
display(history_df.round(4))

val_df = pd.DataFrame({
    'reviews': val_texts,          # same order as val_loader (it is not shuffled)
    'true': best_truth,
    'predicted': best_preds,
    'prob_positive': best_probs,
})
val_df['correct'] = val_df['true'] == val_df['predicted']
val_df['confidence'] = np.where(val_df['predicted'] == 1,
                                val_df['prob_positive'], 1 - val_df['prob_positive'])
val_df.to_csv(RESULTS_DIR / 'distilbert_val_predictions.csv', index=False, lineterminator='\n')
print(f"{(~val_df['correct']).sum()} wrong out of {len(val_df)} validation reviews")

## 6c. Figures

Same palette and paper colour as every other project figure (negative `#2a78d6`,
positive `#eb6834`, paper `#fcfcfb`). Saved at 200 dpi into `results/figures/`.

1. **Learning curves** - loss and validation scores per epoch, for question (c).
   The dotted line is the Linear SVM's 0.8579 macro F1, the model to beat.
2. **Confusion matrix** - shaded by share of each true class, like `fig07`.
3. **Confidence of right vs wrong predictions** - for question (e): a good model is
   confident when right and unsure when wrong.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

NEG, POS = '#2a78d6', '#eb6834'
INK, INK_MUTED, AXIS, PAPER = '#0b0b0b', '#898781', '#c3c2b7', '#fcfcfb'
SVM_MACRO_F1 = 0.8579   # Linear SVM on the same 20% split - src/compare_models.py
plt.rcParams.update({
    'savefig.dpi': 200, 'figure.facecolor': PAPER, 'axes.facecolor': PAPER,
    'font.size': 10, 'axes.edgecolor': AXIS, 'axes.labelcolor': INK_MUTED,
    'xtick.color': INK_MUTED, 'ytick.color': INK_MUTED, 'text.color': INK,
    'axes.spines.top': False, 'axes.spines.right': False,
})
epochs = history_df['epoch']

# 1. Learning curves: two panels, one y-scale each (loss vs scores never share an axis)
fig, (ax_loss, ax_score) = plt.subplots(1, 2, figsize=(10, 3.8))
ax_loss.plot(epochs, history_df['train_loss'], color=INK, lw=2, marker='o', ms=6)
ax_loss.plot(epochs, history_df['val_loss'], color=INK_MUTED, lw=2, marker='o', ms=6, ls='--')
ax_loss.annotate('train', (epochs.iloc[-1], history_df['train_loss'].iloc[-1]),
                 xytext=(6, 0), textcoords='offset points', va='center', color=INK)
ax_loss.annotate('validation', (epochs.iloc[-1], history_df['val_loss'].iloc[-1]),
                 xytext=(6, 0), textcoords='offset points', va='center', color=INK_MUTED)
ax_loss.set_title('Cross-entropy loss', loc='left', fontweight='bold')
ax_loss.set_xlabel('epoch'); ax_loss.set_ylabel('loss')

for col, colour, style, label in [
    ('val_accuracy', INK_MUTED, '--', 'accuracy'),
    ('val_macro_f1', INK, '-', 'macro F1'),
    ('val_negative_f1', NEG, '-', 'negative F1'),
]:
    ax_score.plot(epochs, history_df[col], color=colour, lw=2, marker='o', ms=6, ls=style)
    ax_score.annotate(f'{label} {history_df[col].iloc[-1]:.3f}',
                      (epochs.iloc[-1], history_df[col].iloc[-1]),
                      xytext=(6, 0), textcoords='offset points', va='center', color=INK)
ax_score.axhline(SVM_MACRO_F1, color=INK_MUTED, lw=1, ls=':')
ax_score.annotate(f'Linear SVM macro F1 {SVM_MACRO_F1}', (epochs.iloc[0], SVM_MACRO_F1),
                  xytext=(0, -12), textcoords='offset points', color=INK_MUTED, fontsize=8)
ax_score.axvline(best_epoch, color=AXIS, lw=1)
ax_score.set_title(f'Validation scores (best epoch {best_epoch})', loc='left', fontweight='bold')
ax_score.set_xlabel('epoch'); ax_score.set_ylabel('score')
for ax in (ax_loss, ax_score):
    ax.set_xticks(list(epochs)); ax.set_xlim(0.7, len(epochs) + 1.3)
    ax.yaxis.grid(True, color=AXIS, lw=0.6, alpha=0.5)
fig.tight_layout()
fig.savefig(FIG_DIR / 'distilbert_learning_curves.png', bbox_inches='tight')
plt.show()

# 2. Confusion matrix, shaded by share of each true class
share_cmap = LinearSegmentedColormap.from_list('share', [PAPER, NEG])
shares = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(4, 3.6))
ax.imshow(shares, cmap=share_cmap, vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        dark = shares[i, j] > 0.55
        ax.text(j, i - 0.08, f'{cm[i, j]:,}', ha='center', va='center',
                fontsize=13, fontweight='bold', color=PAPER if dark else INK)
        ax.text(j, i + 0.2, f'{shares[i, j]:.0%} of true class', ha='center',
                va='center', fontsize=7.6, color=PAPER if dark else INK_MUTED)
ax.set_xticks([0, 1], ['pred neg', 'pred pos']); ax.set_yticks([0, 1], ['true neg', 'true pos'])
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title('DistilBERT, 20% validation split', loc='left', fontweight='bold')
fig.tight_layout()
fig.savefig(FIG_DIR / 'distilbert_confusion_matrix.png', bbox_inches='tight')
plt.show()

# 3. Confidence of right vs wrong predictions: small multiples, shared x-axis
bins = np.linspace(0.5, 1.0, 21)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharex=True)
for ax, ok, title in [(axes[0], True, 'Correct'), (axes[1], False, 'Wrong')]:
    part = val_df.loc[val_df['correct'] == ok, 'confidence']
    ax.hist(part, bins=bins, color=INK_MUTED, edgecolor=PAPER, linewidth=1)
    ax.set_title(f'{title} predictions (n = {len(part)}, median {part.median():.2f})',
                 loc='left', fontweight='bold', fontsize=10)
    ax.set_xlabel('model confidence in its prediction'); ax.set_ylabel('reviews')
    ax.yaxis.grid(True, color=AXIS, lw=0.6, alpha=0.5)
fig.tight_layout()
fig.savefig(FIG_DIR / 'distilbert_confidence.png', bbox_inches='tight')
plt.show()

## 6d. Error-analysis material for question (e)

Prints DistilBERT's most confident mistakes, correctly handled negative reviews
that contain a negation, and **what DistilBERT predicts for the cases already in
the report's Section 6**. The report currently says we "did not check
DistilBERT's predictions on these specific reviews" - this answers that.

In [ ]:
pd.set_option('display.max_colwidth', 300)
LABEL = {0: 'negative', 1: 'positive'}

def show(df, n=5):
    for _, r in df.head(n).iterrows():
        print(f"true {LABEL[r['true']]:<8} | predicted {LABEL[r['predicted']]:<8} "
              f"| P(positive) {r['prob_positive']:.3f}")
        print('   ', r['reviews'][:400].replace('\n', ' '), '\n')

print('=== Most confident MISTAKES ===')
show(val_df[~val_df['correct']].sort_values('confidence', ascending=False))

print('=== Correct NEGATIVE reviews that contain a negation (a TF-IDF weak spot) ===')
negation = val_df['reviews'].str.contains(r"\b(?:not|never|no|n't)\b", case=False, regex=True)
show(val_df[val_df['correct'] & (val_df['true'] == 0) & negation]
     .sort_values('confidence', ascending=False))

print('=== The cases quoted in the report (Section 6), as DistilBERT sees them ===')
REPORT_CASES = {
    'Case 1 (likely wrong label)': 'Dilvered quickly',
    'Case 1b (likely wrong label)': 'Saucony Jazz',
    'Case 2 (product vs service)': 'It took them weeks',
    'Case 3 (misspelling)': 'a little dissapointed',
}
for name, snippet in REPORT_CASES.items():
    hit = val_df[val_df['reviews'].str.contains(snippet, regex=False)]
    if hit.empty:
        print(f'{name}: not found in the validation split')
    for _, r in hit.iterrows():
        print(f"{name}: true {LABEL[r['true']]}, DistilBERT predicts {LABEL[r['predicted']]} "
              f"(P(positive) = {r['prob_positive']:.3f})")

## 7. Write `results/distilbert_metrics.json`

This is the handoff. `src/compare_models.py` prefers this file over scraping
`03_distilbert.ipynb`, and because `test_size` is 0.2 it marks the row
`comparable: true` - so DistilBERT joins the main table and
`fig08_model_comparison.png` loses its hatching.

After running this, back in the repo:

```bash
python src/compare_models.py
python src/make_comparison_figures.py
```

In [ ]:
metrics = {
    'name': 'DistilBERT (4 epochs, max_length=256)',
    'short': 'DistilBERT',
    'owner': 'Ryan',
    'source': 'notebooks/04_distilbert_20split.ipynb',
    'test_size': TEST_SIZE,
    'accuracy': float(accuracy_score(best_truth, best_preds)),
    'macro_f1': float(f1_score(best_truth, best_preds, average='macro')),
    'negative_precision': report['negative']['precision'],
    'negative_recall': report['negative']['recall'],
    'negative_f1': report['negative']['f1-score'],
    'positive_precision': report['positive']['precision'],
    'positive_recall': report['positive']['recall'],
    'positive_f1': report['positive']['f1-score'],
    'confusion_matrix': {
        'tn': int(cm[0, 0]), 'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]), 'tp': int(cm[1, 1]),
    },
    'train_seconds': round(train_seconds, 1),
    'val_seconds_per_epoch': history[-1]['val_seconds'],
    'history': history,
    'parameters': int(N_PARAMS),
    'config': {
        'model': MODEL_NAME,
        'max_length': MAX_LEN,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'warmup': '10% of total steps',
        'device': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu',
        'peak_gpu_memory_gb': (round(torch.cuda.max_memory_allocated() / 1e9, 2)
                               if DEVICE.type == 'cuda' else None),
        'best_epoch': best_epoch,
    },
    'why': ('Contextual subword embeddings instead of a bag of words, so negation '
            'and word order survive, at far higher parameter and compute cost.'),
}

out_path = RESULTS_DIR / 'distilbert_metrics.json'
with open(out_path, 'w', encoding='utf-8', newline='\n') as f:
    json.dump(metrics, f, indent=2)
    f.write('\n')
print('wrote', out_path)
print(json.dumps(metrics, indent=2)[:600])

## 8. Test predictions

Written to `results/submission_distilbert_20split.csv` so it never silently
overwrites either `submission.csv` or Ryan's original
`submission_distilbert.csv`. Deciding which file is submitted is a group call.

In [ ]:
model.load_state_dict(torch.load(BEST_PATH))
model.eval()

test_predictions, test_probs = [], []
test_started = time.perf_counter()
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test'):
        logits = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
        ).logits
        test_probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy().tolist())
        test_predictions.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
test_seconds = time.perf_counter() - test_started

submission = pd.DataFrame({
    'reviews': test_df['reviews'],
    'predictions': [int(p) for p in test_predictions],   # the course column name
})
sub_path = RESULTS_DIR / 'submission_distilbert_20split.csv'
submission.to_csv(sub_path, index=False, lineterminator='\n')
print('wrote', sub_path, '|', len(submission), 'rows')
print(submission['predictions'].value_counts().to_dict())

# Probabilities go in a separate file so the submission keeps the course format.
# Question (e) asks about TEST samples: read some of these and judge them by hand.
test_out = submission.assign(prob_positive=test_probs)
test_out.to_csv(RESULTS_DIR / 'distilbert_test_probabilities.csv', index=False, lineterminator='\n')

# Inference cost for question (f), added to the metrics file written above.
metrics['test_seconds'] = round(test_seconds, 1)
metrics['test_reviews_per_second'] = round(len(submission) / test_seconds, 1)
with open(out_path, 'w', encoding='utf-8', newline='\n') as f:
    json.dump(metrics, f, indent=2)
    f.write('\n')
print(f'test inference: {test_seconds:.1f}s for {len(submission)} reviews '
      f'({metrics["test_reviews_per_second"]} reviews/s)')

print('\n=== Least confident TEST predictions - good candidates to discuss in (e) ===')
test_out['confidence'] = np.maximum(test_out['prob_positive'], 1 - test_out['prob_positive'])
for _, r in test_out.sort_values('confidence').head(5).iterrows():
    print(f"predicted {'positive' if r['predictions'] else 'negative'} "
          f"(P(positive) = {r['prob_positive']:.3f})")
    print('   ', r['reviews'][:400].replace('\n', ' '), '\n')

## 9. Download the results

Colab deletes its files when the session ends, so this step is essential. It
downloads one zip of `results/` holding:

| File | For |
|---|---|
| `distilbert_metrics.json` | `src/compare_models.py` - the main comparison table, (d) and (f) |
| `distilbert_history.csv` | per-epoch loss and scores - question (c) |
| `distilbert_val_predictions.csv` | every validation review with its probability - question (e) |
| `distilbert_test_probabilities.csv` | test predictions with probabilities - question (e) |
| `submission_distilbert_20split.csv` | the candidate `submission.csv` - question (d) |
| `figures/distilbert_*.png` | learning curves, confusion matrix, confidence |

Also use `File` -> `Download` -> `Download .ipynb` to keep the printed outputs as
evidence.

In [ ]:
import shutil

archive = shutil.make_archive('distilbert_20split_results', 'zip', RESULTS_DIR)
print('packed', archive)
if IN_COLAB:
    files.download(archive)